# Grid check - does an 8 px grid help, before any 8 px student is trained

Stage A's students lose to the frozen satellite DINOv3 by 0.03 to 0.06 on every radar row. The planned 8 px run (a student with a finer grid, every model scored again at 8 px) would cost 10 to 13 A100 hours. This notebook asks the question behind it for about half an A100 hour - does a finer feature grid help the scoring at all, for the strong frozen model. The rule was written down before it runs (2026-10-04).

## What runs

The frozen satellite DINOv3 ViT-L at layer 21 with part two's three channel input, on glacial lakes and forest, the validation split, the decoder with the seeds 0, 1 and 2, three settings in one session

- 16 px, the run step as it is (every saved row was made this way)
- AnyUp 8 px, the features made twice as fine by AnyUp, guided by the decoder's grey radar image at the chip's full size, the probe on 8 px patches and the decoder entering at 8 px (the same decoder without its first stage)
- bilinear 8 px, the same with plain bilinear upsampling, the control

The encoder is loaded once and the features of each split are extracted once for the three settings.

## The rule, written before the run

The decoder IoU of the lake class on val. If AnyUp 8 px beats 16 px by the rule (a gap larger than 0.01 and larger than twice the larger seed SD), the finer grid helps and a cut-down 8 px run is proposed with its compute. Otherwise the 8 px run and our own upsampler are dropped and part four is written up. Forest, the probe rows and bilinear are reported and do not decide.

What is known before it runs. In part one (calving fronts, the same DINOv3) no AnyUp head beat the 16 px decoder with its image skip. In part two AnyUp features kept the classes less far apart on forest, lakes and alpine glaciers.

## Runtime and time

An A100, High-RAM is not needed. Either shape Colab gives is enough, a 40 GB GPU with 83.5 GB of RAM or an 80 GB GPU with 179 GB. The peak RAM is about 65 GB by arithmetic, the lake features at 8 px are 31 GB for the 14,820 training chips, the 16 px ones kept next to them 9 GB and the decoder's 4,000 chips 8 GB (the probe copies only its 2 million sampled patches). About 30 minutes, an estimate - the features of lakes (256 px chips) and forest (128 px) once, AnyUp and bilinear on them, three probes and nine decoders per dataset at 256 and 128 px.

## Data and the .env file

The two datasets as notebook 03 takes them, from their private Hugging Face copies when DATASET_HF_OWNER is set, else from their copies on Drive under MyDrive/datasets, the train and val splits only. The AnyUp weights (3.5 MB, CC BY 4.0) are a file on Drive under MyDrive/sar-transfer-distillation/weights, checked by size and hash and never downloaded. HF_TOKEN reads the gated DINOv3 weights and is never printed.

## Results

Google Drive, folder MyDrive/sar-transfer-distillation/grid_check/. The files of the run step for each dataset and setting and grid_check.json with the verdict and every compared row.

## How to run

Run the cells from top to bottom. A setting whose summary is on Drive is not run again. The last cell flushes Drive and then releases the Colab machine (set END_RUNTIME = False in that cell to keep the runtime), and a cell that fails stops the run before it.


In [ ]:
# Mount Google Drive. The data and the results are there.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# Setup. Tokens, model cache, GPU check, folders, the two datasets on the runtime disk, the AnyUp weights.
import json
import os
import time
from pathlib import Path

import pandas as pd
import torch

from sartransfer import env
from sartransfer.distil import score, stage0
from sartransfer.domains import hub, prepared, upsample
from sartransfer.domains import run as run_step
from sartransfer.domains.prepared import load_manifest
from sartransfer.models import anyup_loader

start = time.perf_counter()
assert torch.cuda.is_available(), "no GPU, pick the A100 runtime and connect again"
DEVICE = "cuda"
MODEL_CACHE = env.model_cache_on_drive()       # before transformers or huggingface_hub load
assert MODEL_CACHE is not None, "Google Drive is not mounted, run the mount cell first"
env.load_env()
HF_TOKEN = env.require("HF_TOKEN", "the gated DINOv3 weights and the private data copies")
assert HF_TOKEN, "no Hugging Face token was given"
DATA_TOKEN = os.environ.get("HF_WRITE_TOKEN") or HF_TOKEN   # the private dataset copies, as notebook 03
DATASET_OWNER = os.environ.get("DATASET_HF_OWNER", "").strip()   # optional, without it the Drive copies are used

# The test, fixed before it runs
DATASETS = ("glacial_lakes", "forty_v1")            # lakes decides, forest is reported
DECIDES = "glacial_lakes"
METRIC = {"glacial_lakes": "IoU_pos", "forty_v1": "mIoU"}   # as stage A's table, the lake IoU and forest mIoU
ENCODER, MODE = "dinov3-l-sat", "vv_vh_diff"        # part two's run of the frozen satellite DINOv3
LAYER = run_step.PRE_REGISTERED_LAYERS[ENCODER]     # 21, as part two
SPLITS = ("val",)                                   # stage A, the validation split
SEEDS = (0, 1, 2)                                   # the decoder seeds of every saved row
NEEDED = ("train", *SPLITS)

CONTENT, MYDRIVE = (Path(folder) for folder in stage0.WORK_FOLDERS)
PREP = CONTENT / "sar-transfer-distillation" / "prepared"          # runtime disk, as notebook 03
OUT = MYDRIVE / "sar-transfer-distillation" / "grid_check"         # the results of this notebook
WEIGHTS = MYDRIVE / "sar-transfer-distillation" / "weights" / anyup_loader.WEIGHTS_NAME
TIMES = {}                       # seconds of every step of this run, by name


def took(since, name):
    """The time of a cell as text, recorded under `name`."""
    TIMES[name] = time.perf_counter() - since
    return f"cell time {TIMES[name]:.1f} s"


def hide(text):
    """A text without the tokens."""
    out = str(text)
    for secret in (HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")):
        if secret:
            out = out.replace(secret, "<token>")
    return out


def on_disk(name):
    """True when the manifest and the train and val files of a dataset are on the runtime disk."""
    if not (PREP / name / "manifest.json").is_file():
        return False
    return all((PREP / name / file).is_file() for file in prepared.split_files(NEEDED))


for name in DATASETS:
    if on_disk(name):
        print(f"{name}  on the runtime disk already")
    else:
        clock, source = time.perf_counter(), None
        if DATASET_OWNER:
            try:                                   # first the private Hugging Face copy, train and val
                hub.fetch(name, DATASET_OWNER, PREP, token=DATA_TOKEN, splits=NEEDED)
                source = "Hugging Face"
            except Exception as err:
                print(f"{name}  not on Hugging Face ({type(err).__name__}, {hide(err)[:200]})")
        if source is None:                         # then the copy on Drive
            copy = MYDRIVE / "datasets" / hub.repo_name(name)
            assert (copy / "manifest.json").is_file(), f"{name} is neither on Hugging Face nor on Drive under {copy}"
            prepared.copy_prepared(copy, PREP, name, splits=NEEDED)
            source = "Drive"
        print(f"{name}  from {source} in {(time.perf_counter() - clock) / 60:.1f} min")
    man = load_manifest(PREP, name)
    print(f"{name:<14} splits {man['splits']}, {man['chip_size']} px, units {man['units']}")
assert WEIGHTS.is_file(), f"{WEIGHTS} is not on Drive, the AnyUp weights of this notebook"
print(f"AnyUp weights  {WEIGHTS}")
print(f"GPU     {torch.cuda.get_device_name(0)}")
print(f"results go to   {OUT}")
OUT.mkdir(parents=True, exist_ok=True)
print(took(start, "setup"))


In [ ]:
# Runs. Per dataset the three settings, the 16 px run step first, then AnyUp and bilinear at 8 px
# (run_domain's upsampler). The encoder is loaded once and the features of each split are extracted once and
# kept for the three settings (enc_cache and feature_cache), so the settings differ only in the grid. A setting
# whose summary is on Drive is not run again.
start = time.perf_counter()
UPSAMPLERS = {"16 px": None, "AnyUp 8 px": upsample.AnyUp(WEIGHTS, DEVICE), "bilinear 8 px": upsample.Bilinear(DEVICE)}
enc_cache = {}
try:
    for name in DATASETS:
        feature_cache = {}                         # the features of this dataset, for its three settings
        for setting, up in UPSAMPLERS.items():
            print(f"\n---- {name}, {setting}")
            run_step.run_domain(PREP, name, ENCODER, OUT, token=HF_TOKEN, layer=LAYER, seeds=SEEDS, mode=MODE,
                                device=DEVICE, splits=SPLITS, enc_cache=enc_cache, feature_cache=feature_cache,
                                upsampler=up)
        feature_cache.clear()                      # the next dataset needs the RAM
finally:
    enc_cache.clear()
    torch.cuda.empty_cache()
print(took(start, "runs"))


In [ ]:
# Verdict. The decoder and probe rows of each setting (seed mean and SD on val), then each 8 px setting against
# 16 px by the rule (score.judge). The word follows the rule written before the run
# - the AnyUp 8 px decoder against the 16 px one on the lake IoU. Forest, the probe rows and
# bilinear are reported. Everything goes to grid_check.json on Drive.
start = time.perf_counter()


def run_of(name, up):
    """The run name run_domain gives a dataset and setting."""
    return score.run_name(name, ENCODER, LAYER, MODE) + ("" if up is None else f"__up{run_step.PATCH // up.factor}_{up.name}")


rows = []
for name in DATASETS:
    for setting, up in UPSAMPLERS.items():
        summary = pd.read_csv(OUT / f"{run_of(name, up)}__summary.csv")
        summary = summary[summary["split"] == SPLITS[0]]
        for head in ("decoder", "probe"):
            values = summary.loc[summary["head"] == head, METRIC[name]]
            rows.append({"dataset": name, "setting": setting, "head": head, "metric": METRIC[name],
                         "mean": float(values.mean()), "sd": float(values.std(ddof=1)) if len(values) > 1 else float("nan"),
                         "seeds": len(values)})
table = pd.DataFrame(rows)
print("every setting, the validation split")
print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
compared = []
for name in DATASETS:
    for head in ("decoder", "probe"):
        ref = table[(table["dataset"] == name) & (table["setting"] == "16 px") & (table["head"] == head)].iloc[0]
        for setting in ("AnyUp 8 px", "bilinear 8 px"):
            row = table[(table["dataset"] == name) & (table["setting"] == setting) & (table["head"] == head)].iloc[0]
            outcome, gap, needed = score.judge(head, row["mean"], row["sd"], ref["mean"], ref["sd"])
            compared.append({"dataset": name, "head": head, "setting": setting, "metric": METRIC[name],
                             "score": row["mean"], "score_16px": ref["mean"], "gap": gap, "needed": needed,
                             "outcome": outcome, "decides": name == DECIDES and head == "decoder" and setting == "AnyUp 8 px"})
compared = pd.DataFrame(compared)
print("\neach 8 px setting against 16 px (gap = 8 px - 16 px, needed = the margin the gap must pass)")
print(compared.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
deciding = compared[compared["decides"]].iloc[0]
if deciding["outcome"] == "win":
    verdict = "helps"
    words = (f"THE FINER GRID HELPS - AnyUp 8 px beats 16 px on the lake IoU by {deciding['gap']:.4f} (needed "
             f"{deciding['needed']:.4f}). A cut-down 8 px run is proposed with its compute")
elif deciding["outcome"] == "missing":
    verdict = "not known"
    words = "NOT KNOWN - a lake decoder row is missing, see the lines above. Nothing is decided"
else:
    verdict = "no gain"
    words = (f"NO GAIN - AnyUp 8 px against 16 px on the lake IoU is a {deciding['outcome']} (gap {deciding['gap']:.4f}, "
             f"needed {deciding['needed']:.4f}). The 8 px run and our own upsampler are dropped, part four is written up")
print(f"\n{words}")
REPORT = {"verdict": verdict, "words": words, "encoder": ENCODER, "layer": LAYER, "mode": MODE, "split": SPLITS[0],
          "seeds": list(SEEDS), "gpu": torch.cuda.get_device_name(0), "settings": table.to_dict("records"),
          "compared": compared.to_dict("records"), "times": TIMES}
(OUT / "grid_check.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")
print(f"saved {OUT / 'grid_check.json'}")
print(took(start, "verdict"))


In [ ]:
# Flush Drive and release the runtime. The run is over when this cell prints, Drive then holds
# everything that was written. With END_RUNTIME the Colab machine is released right after, so
# the GPU stops and no compute is spent waiting - google.colab.runtime.unassign asks Colab from
# the machine itself (an HTTP request to the runtime service, read in the colabtools source on
# 2026-10-04), so it works from VS Code as well as from the browser. A cell that fails earlier
# stops the run before this one, so a runtime with an error stays up to be looked at. Set
# END_RUNTIME = False to keep the runtime; run the mount cell again to use Drive afterwards.
from google.colab import drive, runtime

END_RUNTIME = True               # release the machine after the flush, the outputs stay in the notebook

drive.flush_and_unmount()
print("Drive flushed and unmounted, the run is over"
      + (", the runtime is released now" if END_RUNTIME else ", the runtime stays up"), flush=True)
if END_RUNTIME:
    runtime.unassign()
